In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
classes = os.listdir(os.path.join(path))
print(classes)

In [ ]:
from torchvision.datasets import ImageFolder
import torch

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

dataset = ImageFolder(root=os.path.join(path, "dataset"))
print(dataset.classes)

In [ ]:
train_image_dir = os.path.join(path, "train", "images")
print(train_image_dir)

In [ ]:

dataset = ImageFolder(root=os.path.join(path, "dataset/masks"))
classes = os.listdir(os.path.join(dataset))


In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

class customDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform

        # Get image files sorted for consistency
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))])


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)
        else:
          image = transforms.ToTensor()(image)
          mask = torch.tensor(np.array(mask), dtype=torch.long)

        mask = remap_mask(mask)

        return image, mask  # Return image-mask pair

In [ ]:
from torch.utils.data import DataLoader
from torch import nn

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),
])

In [ ]:
train_image_dir = os.path.join(path, "train", "images")
train_mask_dir = os.path.join(path, "train", "masks")

test_image_dir = os.path.join(path, "val", "images")
test_mask_dir = os.path.join(path, "val", "masks")


train_dataset = customDataset(train_image_dir, train_mask_dir, transform=image_transforms, target_transform=mask_transforms)
test_dataset = customDataset(test_image_dir, test_mask_dir, transform=image_transforms, target_transform=mask_transforms)


# Create Train & Test DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

In [ ]:
import matplotlib.pyplot as plt


def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()



In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
).to(device)

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TO DO

def train_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        #forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)

        #backaard
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)

    return total_loss / len(train_loader.dataset)

def validate(model, val_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item() * images.size(0)


    return  total_loss / len(val_loader.dataset)

In [ ]:
# TO DO
import torch
from torch import nn
import torch.optim as optim
import matplotlib.pyplot as plt


loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

num_epochs = 2
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate(model, val_losses, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# Plot loss curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
model.eval()

# Get predictions on validation set
fig, axes = plt.subplots(5, 3)

with torch.no_grad():
    for idx in range(5):
        image, mask = val_dataset[idx]
        image_batch = image.unsqueeze(0).to(device)


plt.show()